<a href="https://colab.research.google.com/github/s-araromi/flyrank-ml-internship-sulaimon/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-araromi/flyrank-ml-internship-sulaimon/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring

**Primary task type:** Ranking/scoring

The decision is: **Which content pages should be reviewed first?**

The output will be a ranked review queue for a content editor, SEO analyst, or content strategist. Pages with higher scores will be reviewed first so that the decision-maker can choose whether to refresh, optimise, investigate, or continue monitoring them.

This is primarily a ranking/scoring task rather than a simple classification task. A binary decline label can help with training and evaluation, but the operational output must order pages by priority because the review team has limited time and cannot inspect every page at once.

In [6]:
from pathlib import Path
import os
import subprocess
import pandas as pd

REPO_URL = "https://github.com/s-araromi/flyrank-ml-internship-sulaimon.git"
PROJECT_ROOT = Path("/content/flyrank-ml-internship-sulaimon")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)],
        check=True
    )

os.chdir(PROJECT_ROOT)

DATA_PATH = PROJECT_ROOT / "data/raw/content_refresh_anonymized.csv"
assert DATA_PATH.exists(), f"Starter dataset not found: {DATA_PATH}"

task_frame = pd.DataFrame(
    {
        "lane": ["Refresh / Content Opportunity Scoring"],
        "task_type": ["Ranking / scoring"],
        "decision": ["Which content pages should be reviewed first?"],
        "decision_maker": ["Content editor, SEO analyst, or content strategist"],
        "output": ["Ranked content-review queue"],
    }
)

print(f"Working folder: {PROJECT_ROOT}")
print(f"Starter dataset found: {DATA_PATH.relative_to(PROJECT_ROOT)}")
display(task_frame)

Working folder: /content/flyrank-ml-internship-sulaimon
Starter dataset found: data/raw/content_refresh_anonymized.csv


,lane,task_type,decision,decision_maker,output
0,Refresh / Content Opportunity Scoring,Ranking / scoring,Which content pages should be reviewed first?,"Content editor, SEO analyst, or content strate...",Ranked content-review queue


## 2. Target or proxy

**Proposed proxy target:** `target_declining_proxy`

The starter CSV does not contain a ready-made target column. For this framing exercise, I define `target_declining_proxy` as:

- `1` when the page's observed `trend_direction` is `down`.
- `0` otherwise.

This proxy identifies pages currently showing an observed downward trend. The operational system would use an estimated probability or opportunity score to rank pages for human review.

This is a rule-defined proxy based on the same trailing 90-day snapshot, not an independently observed future outcome. It must therefore not be described as proof that a page will decline in the future. A stronger later model should define its target from performance measured in a separate future outcome window.

Because the proxy is constructed from `trend_direction`, both `trend_direction` and `trend_pct` are outcome-construction fields and must not be used as predictors. The pseudonymized identifiers `content_id` and `client_id` must also be excluded from predictors; they are only suitable for tracing, grouping, and client-held-out validation.

**False-positive cost:** an editor spends time reviewing or updating a page that did not require immediate attention.

**False-negative cost:** a genuinely declining or high-opportunity page is not prioritised and may continue losing visibility or traffic.

In [7]:
# Load the starter dataset and construct the proposed proxy target.
content_df = pd.read_csv(DATA_PATH)

required_target_columns = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
}

missing_target_columns = required_target_columns.difference(content_df.columns)
assert not missing_target_columns, (
    f"Missing required columns: {sorted(missing_target_columns)}"
)

# Inspect the observed categories before constructing the proxy.
trend_categories = sorted(
    content_df["trend_direction"]
    .dropna()
    .astype(str)
    .str.lower()
    .unique()
)

print(f"Observed trend categories: {trend_categories}")
assert "down" in trend_categories, (
    "Expected the trend_direction column to contain the category 'down'."
)

# This is a rule-defined proxy for framing, not a future-outcome label.
content_df["target_declining_proxy"] = (
    content_df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype("int8")
)

target_summary = (
    content_df["target_declining_proxy"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("target_declining_proxy")
    .reset_index(name="pages")
)

target_summary["meaning"] = target_summary["target_declining_proxy"].map(
    {
        0: "Not observed as declining",
        1: "Observed as declining",
    }
)

target_summary["percentage"] = (
    100 * target_summary["pages"] / len(content_df)
).round(1)

target_preview = content_df[
    [
        "content_id",
        "target_declining_proxy",
        "trend_direction",
        "trend_pct",
    ]
].head(8)

print(f"Rows loaded: {len(content_df):,}")
print("Proxy created: target_declining_proxy")
print(
    "Leakage warning: trend_direction and trend_pct define the proxy "
    "and cannot be used as predictors."
)

display(target_summary)
display(target_preview)

Observed trend categories: ['down', 'flat', 'new', 'stable', 'up']
Rows loaded: 30,000
Proxy created: target_declining_proxy
Leakage warning: trend_direction and trend_pct define the proxy and cannot be used as predictors.


,target_declining_proxy,pages,meaning,percentage
0,0,13738,Not observed as declining,45.8
1,1,16262,Observed as declining,54.2


,content_id,target_declining_proxy,trend_direction,trend_pct
0,content_304f48230142,1,down,-41.4
1,content_a1fb4e703a9e,1,down,-57.7
2,content_9aa793d4d895,1,down,-60.9
3,content_331d6c4de07b,0,stable,-13.8
4,content_d99b7a2d90ca,1,down,-34.7
5,content_d4084a4bc775,1,down,-38.9
6,content_9a34b442b552,1,down,-92.3
7,content_a63219c6e95a,0,stable,0.6


## 3. Success metric

**Primary success metric:** Precision@50

Precision@50 is the proportion of pages with `target_declining_proxy = 1` among the 50 pages receiving the highest priority scores.

\[
\text{Precision@50} =
\frac{\text{declining pages among the top 50 ranked pages}}{50}
\]

This metric matches the operational decision because the content team has limited review capacity and needs the first 50 pages in the queue to contain as many genuinely declining pages as possible. A false positive consumes a review slot, so precision at the review cutoff is more useful than overall accuracy.

For this project, I will regard performance as useful when:

1. Precision@50 is at least **0.70**, meaning at least 35 of the first 50 pages are observed as declining;
2. it exceeds the transparent fixed-rule baseline; and
3. it is evaluated on clients excluded from model training.

The proxy prevalence is approximately 0.542, so selecting pages without useful ranking information would be expected to find about 27 declining pages among 50. Reaching 35 or more would represent a practically meaningful improvement in the review queue.

This metric evaluates agreement with the current decline proxy. It does not prove that reviewing a page will improve traffic, and it does not measure future decline until a time-separated outcome is introduced.

In [8]:
import math

queue_size = 50
desired_precision_at_50 = 0.70

proxy_prevalence = content_df["target_declining_proxy"].mean()
minimum_relevant_pages = math.ceil(
    queue_size * desired_precision_at_50
)
expected_relevant_at_prevalence = queue_size * proxy_prevalence
required_absolute_lift = (
    desired_precision_at_50 - proxy_prevalence
)

metric_plan = pd.DataFrame(
    {
        "metric_component": [
            "Review queue size (K)",
            "Desired Precision@50",
            "Minimum declining pages required",
            "Proxy prevalence reference",
            "Expected declining pages at prevalence",
            "Required absolute lift over prevalence",
        ],
        "value": [
            queue_size,
            desired_precision_at_50,
            minimum_relevant_pages,
            round(proxy_prevalence, 3),
            round(expected_relevant_at_prevalence, 1),
            round(required_absolute_lift, 3),
        ],
    }
)

print("Primary metric: Precision@50")
print(
    f"Success threshold: at least {minimum_relevant_pages} "
    f"declining pages among the top {queue_size}."
)
print(
    f"Proxy prevalence reference: {proxy_prevalence:.3f} "
    f"({proxy_prevalence:.1%})."
)
print(
    "Final comparison requirement: outperform a transparent fixed-rule "
    "baseline on client-held-out data."
)

display(metric_plan)

Primary metric: Precision@50
Success threshold: at least 35 declining pages among the top 50.
Proxy prevalence reference: 0.542 (54.2%).
Final comparison requirement: outperform a transparent fixed-rule baseline on client-held-out data.


,metric_component,value
0,Review queue size (K),50.000
1,Desired Precision@50,0.700
2,Minimum declining pages required,35.000
3,Proxy prevalence reference,0.542
4,Expected declining pages at prevalence,27.100
5,Required absolute lift over prevalence,0.158


## 4. The unit of analysis, as a real dataframe

**Unit of analysis:** one pseudonymized content page.

Each row represents one content page belonging to one pseudonymized client. The dataset contains 30,000 pages across 32 clients, and `content_id` should be unique at this grain.

The lane dataframe includes candidate signals describing:

- search visibility and traffic;
- engagement;
- keyword opportunity;
- content age and freshness; and
- the constructed decline proxy.

`content_id` and `client_id` are retained only for traceability, grouping, and client-held-out validation. They will not be used as predictors.

The outcome-construction columns `trend_direction` and `trend_pct` are intentionally excluded from this modelling slice because they define `target_declining_proxy`. Including them as predictors would create direct data leakage.

Rate columns such as `ctr`, `engagement_rate`, `scroll_rate`, and `ai_traffic_pct` are expressed as percentages multiplied by 100. Missing values must be investigated rather than automatically replaced with zero because missingness may depend on content type. In addition, `avg_position = 0` means that position data are unavailable, not that the page ranked at position zero.

In [9]:
# Construct the lane-specific dataframe at one row per content page.
candidate_feature_columns = [
    "content_type",
    "search_volume",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

lane_columns = [
    "content_id",
    "client_id",
    *candidate_feature_columns,
    "target_declining_proxy",
]

missing_lane_columns = set(lane_columns).difference(content_df.columns)
assert not missing_lane_columns, (
    f"Missing lane columns: {sorted(missing_lane_columns)}"
)

lane_df = content_df[lane_columns].copy()

duplicate_content_ids = lane_df["content_id"].duplicated().sum()

assert len(lane_df) == 30_000, (
    f"Expected 30,000 pages, found {len(lane_df):,}."
)
assert duplicate_content_ids == 0, (
    f"Expected one row per content_id, found "
    f"{duplicate_content_ids:,} duplicated IDs."
)

unit_summary = pd.DataFrame(
    {
        "check": [
            "Rows",
            "Columns in lane slice",
            "Unique content pages",
            "Pseudonymized clients",
            "Duplicated content IDs",
            "Unit of analysis",
        ],
        "result": [
            len(lane_df),
            lane_df.shape[1],
            lane_df["content_id"].nunique(),
            lane_df["client_id"].nunique(),
            duplicate_content_ids,
            "One pseudonymized content page",
        ],
    }
)

missingness_summary = (
    lane_df[candidate_feature_columns]
    .isna()
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
    .rename("missing_percentage")
    .reset_index()
    .rename(columns={"index": "candidate_feature"})
)

print(f"Lane dataframe shape: {lane_df.shape}")
print("Unit of analysis: one pseudonymized content page")
print(
    "Leakage fields excluded from the lane slice: "
    "trend_direction and trend_pct"
)

display(unit_summary)
display(lane_df.head(8))
display(missingness_summary.head(8))

Lane dataframe shape: (30000, 16)
Unit of analysis: one pseudonymized content page
Leakage fields excluded from the lane slice: trend_direction and trend_pct


,check,result
0,Rows,30000
1,Columns in lane slice,16
2,Unique content pages,30000
3,Pseudonymized clients,32
4,Duplicated content IDs,0
5,Unit of analysis,One pseudonymized content page


,content_id,client_id,content_type,search_volume,word_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,target_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,10.0,3221.0,187,20,3803,29,17,0.76,10.6,5.88,4.55,0.0,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,90.0,2481.0,445,25,15320,7,9,0.05,20.3,0.00,10.00,0.0,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.0,3515.0,141,20,12581,11,11,0.09,36.5,0.00,28.57,0.0,1
3,content_331d6c4de07b,client_19581e27de,keyword article,10.0,NaN,463,22,11751,58,78,0.49,6.2,1.28,3.45,0.0,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.0,2803.0,263,14,19140,24,145,0.13,44.0,0.00,24.29,0.0,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,720.0,3080.0,147,20,3970,1,5,0.03,8.5,0.00,25.00,0.0,1
6,content_9a34b442b552,client_8722616204,keyword article,0.0,3059.0,90,20,20,0,1,0.00,7.0,0.00,0.00,0.0,1
7,content_a63219c6e95a,client_19581e27de,keyword article,590.0,NaN,445,22,1724,1,28,0.06,21.2,3.57,7.14,0.0,0


,candidate_feature,missing_percentage
0,word_count,25.7
1,search_volume,8.2
2,scroll_rate,0.4
3,content_age_days,0.0
4,content_type,0.0
5,days_since_last_update,0.0
6,impressions_90d,0.0
7,sessions_90d,0.0


## 5. Why ML beats a fixed rule here

A transparent fixed rule is useful as a baseline, but it is unlikely to be sufficient for the final review queue.

For example, a rule could prioritise pages with at least 500 impressions over 90 days that have not been updated for at least 180 days. This rule is understandable and identifies a small, high-risk subgroup. However, it uses only visibility and freshness thresholds.

The broader prioritisation problem involves several potentially interacting signals:

- search visibility and average position;
- clicks, sessions, and engagement;
- content age and time since the last update;
- keyword opportunity;
- content type;
- systematic missingness; and
- the cost of using limited editorial review capacity.

A single rule cannot easily represent nonlinear relationships, interactions among signals, differences between content types, or the relative priority of all eligible pages. It may also exclude younger pages, lower-volume pages, or pages with unusual combinations of opportunity and risk.

A scoring model may earn its place by combining several non-leaking signals and producing an ordered queue across the wider inventory. However, ML should not be assumed to be better automatically. It should only replace or supplement the fixed rule if it improves client-held-out Precision@50, remains interpretable enough to produce reason codes, and performs consistently across relevant groups.

The output remains decision support for human review. It does not establish that refreshing a page will cause improved traffic or search performance.

In [10]:
# Test the coverage and limitations of one transparent fixed rule.
fixed_rule_mask = (
    (lane_df["impressions_90d"] >= 500)
    & (lane_df["days_since_last_update"] >= 180)
)

fixed_rule_candidates = lane_df.loc[fixed_rule_mask].copy()

rule_candidate_count = len(fixed_rule_candidates)
rule_declining_count = int(
    fixed_rule_candidates["target_declining_proxy"].sum()
)

rule_precision = (
    rule_declining_count / rule_candidate_count
    if rule_candidate_count > 0
    else float("nan")
)

rule_coverage = rule_candidate_count / len(lane_df)
unfilled_queue_slots = max(0, queue_size - rule_candidate_count)

fixed_rule_summary = pd.DataFrame(
    {
        "measure": [
            "Total content pages",
            "Pages selected by fixed rule",
            "Selected pages observed as declining",
            "Proxy precision within selected pages",
            "Inventory coverage",
            "Unfilled positions in a 50-page queue",
        ],
        "value": [
            len(lane_df),
            rule_candidate_count,
            rule_declining_count,
            round(rule_precision, 3),
            round(rule_coverage, 4),
            unfilled_queue_slots,
        ],
    }
)

# Show that content groups differ in scale, outcomes, and missingness.
content_type_summary = (
    content_df.groupby("content_type", dropna=False)
    .agg(
        pages=("content_id", "size"),
        declining_pages=("target_declining_proxy", "sum"),
        decline_rate=("target_declining_proxy", "mean"),
        median_impressions=("impressions_90d", "median"),
        median_days_since_update=("days_since_last_update", "median"),
        word_count_missing_rate=("word_count", lambda values: values.isna().mean()),
    )
    .reset_index()
)

content_type_summary["decline_rate"] = (
    100 * content_type_summary["decline_rate"]
).round(1)

content_type_summary["word_count_missing_rate"] = (
    100 * content_type_summary["word_count_missing_rate"]
).round(1)

content_type_summary = content_type_summary.rename(
    columns={
        "decline_rate": "decline_percentage",
        "word_count_missing_rate": "word_count_missing_percentage",
    }
)

print(
    f"Fixed rule selected {rule_candidate_count:,} of "
    f"{len(lane_df):,} pages."
)
print(
    f"{rule_declining_count:,} selected pages were observed as declining "
    f"(proxy precision = {rule_precision:.3f})."
)
print(
    f"The rule leaves {unfilled_queue_slots} positions unfilled "
    f"in a {queue_size}-page review queue."
)
print(
    "Interpretation: the rule is a useful transparent baseline, "
    "but its narrow coverage cannot produce the complete ranked queue."
)

display(fixed_rule_summary)
display(content_type_summary)

Fixed rule selected 17 of 30,000 pages.
16 selected pages were observed as declining (proxy precision = 0.941).
The rule leaves 33 positions unfilled in a 50-page review queue.
Interpretation: the rule is a useful transparent baseline, but its narrow coverage cannot produce the complete ranked queue.


,measure,value
0,Total content pages,30000.0000
1,Pages selected by fixed rule,17.0000
2,Selected pages observed as declining,16.0000
3,Proxy precision within selected pages,0.9410
4,Inventory coverage,0.0006
5,Unfilled positions in a 50-page queue,33.0000


,content_type,pages,declining_pages,decline_percentage,median_impressions,median_days_since_update,word_count_missing_percentage
0,comparison article,697,399,57.2,107.0,20.0,0.0
1,feedly article,2096,601,28.7,4.0,20.0,0.0
2,keyword article,27207,15262,56.1,955.0,22.0,28.3


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.